# Benchmark Mô hình CKAN trên 3 tập dữ liệu (Movie, Book, Music)
### So sánh: MostPopular vs Item-KNN vs Matrix Factorization vs CKAN

Notebook này chạy thử nghiệm và đánh giá mô hình **CKAN (Collaborative Knowledge-aware Attentive Network)** trên 3 tập dữ liệu:
- **Movie**: MovieLens-1M + Microsoft Satori KG
- **Book**: Book-Crossing + Stanford SNAP/KB KG
- **Music**: Last.FM + KG

### Nội dung thực hiện:
1. **Khám phá dữ liệu (EDA)**: Thống kê số lượng user, item, rating, độ thưa (sparsity), tỉ lệ cold-start và cấu trúc Knowledge Graph.
2. **Baselines**: Cài đặt gọn nhẹ 3 mô hình cơ bản (MostPopular, Item-KNN dùng `scikit-learn`, Matrix Factorization dùng `PyTorch`).
3. **Mô hình CKAN**: Tách rõ từng module độc lập (Sampler, Attention, Aggregator, Model, Trainer, Top-K Evaluator).
4. **Đánh giá trên 3 bài test**:
   - **CTR Prediction**: Đo AUC, F1, Accuracy.
   - **Top-K Ranking**: Đánh giá Recall@K, NDCG@K, Precision@K với K = 5, 10, 20.
   - **Sparsity Test (10% Data)**: Giảm 90% lượng rating để kiểm tra độ ổn định của mô hình khi dữ liệu thưa.
5. **Xuất kết quả**: Bảng thống kê (CSV) và các biểu đồ (PNG) được lưu vào thư mục `./outputs/` để tiện lấy dùng vào báo cáo hoặc đưa vào code khác.



In [ ]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG PHẦN CỨNG & GPU
# ============================================================
import torch

print("=== KIỂM TRA PHẦN CỨNG ===")
print("Phiên bản PyTorch :", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị tính toán:", device)
if torch.cuda.is_available():
    print("Tên GPU           :", torch.cuda.get_device_name(0))
    print("Dung lượng VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")



In [ ]:
# ============================================================
# 2. CÀI ĐẶT THƯ VIỆN & THIẾT LẬP RANDOM SEED
# ============================================================
!pip install -q --upgrade scikit-learn scipy matplotlib seaborn pandas tqdm

import os
import time
import random
import zipfile
import urllib.request
import collections
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#D1D5DB'
plt.rcParams['axes.linewidth'] = 1.0

# Tạo sẵn thư mục lưu kết quả và hình ảnh
os.makedirs("./outputs", exist_ok=True)
print("[OK] Đã sẵn sàng thư viện chuẩn và thiết lập Seed:", SEED)



## 2. Kiến trúc mô hình CKAN

### 2.1. Ý tưởng chính
Lọc cộng tác truyền thống (Matrix Factorization) tính điểm dựa trên tích vô hướng latent vector:
$$y_{u, v} = \sigma(\mathbf{u}_u^T \mathbf{v}_v)$$
Khi ma trận quá thưa (nhiều ô rỗng) hoặc gặp user/item mới (cold-start), mô hình không có đủ tương tác để học tốt.

CKAN giải quyết vấn đề này bằng cách kết hợp Knowledge Graph (KG) theo hai nhánh:
- **User Branch**: Lan truyền sở thích từ các item người dùng từng tương tác sang các entity liên quan trên KG (đạo diễn, diễn viên, thể loại, tác giả...).
- **Item Branch**: Lan truyền thông tin ngữ nghĩa từ item ứng viên sang các láng giềng trên KG.
- **Attention Layer**: Dùng mạng nơ-ron tính attention weight giữa quan hệ và ngữ cảnh tương tác để lọc bớt liên kết nhiễu.

```
                    [ Lịch sử tương tác của User ]                   [ Item ứng viên: v ]
                                 │                                             │
                   0-hop: e_u^(0) = mean(e_items)                 0-hop: e_v^(0) = e_v
                                 │                                             │
                                 ▼                                             ▼
                    [ User Ripple Set: E_u^l ]                    [ Item Ripple Set: E_v^l ]
                          (h, r, t)                                     (h, r, t)
                                 │                                             │
                                 ▼                                             ▼
              ┌────────────────────────────────────┐         ┌────────────────────────────────────┐
              │          ATTENTION LAYER           │         │          ATTENTION LAYER           │
              │   s = MLP([e_h ; e_r])             │         │   s = MLP([e_h ; e_r])             │
              │   alpha = Softmax(s)               │         │   alpha = Softmax(s)               │
              │   e_u^l = sum(alpha * e_t)         │         │   e_v^l = sum(alpha * e_t)         │
              └──────────────────┬─────────────────┘         └──────────────────┬─────────────────┘
                                 │                                             │
                                 ▼                                             ▼
                        [ Aggregator Layer ]                          [ Aggregator Layer ]
                        Vector user: e_u                              Vector item: e_v
                                 └───────────────────────┬─────────────────────┘
                                                         │
                                                         ▼
                                                [ Điểm dự đoán ]
                                           y_hat = Sigmoid(e_u^T * e_v)
```



In [ ]:
# ============================================================
# 3. CẤU HÌNH SIÊU THAM SỐ (HYPERPARAMETERS)
# ============================================================
CONFIG = {
    "movie": {
        "dim": 64,
        "n_layer": 1,
        "itss": 64,
        "utss": 32,
        "lr": 0.002,
        "l2_weight": 1e-5,
        "batch_size": 2048,
        "n_epochs": 10,
        "ratio": 1.0
    },
    "book": {
        "dim": 64,
        "n_layer": 1,
        "itss": 64,
        "utss": 16,
        "lr": 0.001,
        "l2_weight": 1e-5,
        "batch_size": 1024,
        "n_epochs": 8,
        "ratio": 1.0
    },
    "music": {
        "dim": 64,
        "n_layer": 1,
        "itss": 32,
        "utss": 32,
        "lr": 0.002,
        "l2_weight": 1e-5,
        "batch_size": 1024,
        "n_epochs": 10,
        "ratio": 1.0
    }
}

active_datasets = ["movie", "book", "music"]
print("[OK] Đã thiết lập cấu hình tham số cho:", active_datasets)



## 4. Tải và chuẩn bị dữ liệu

Dữ liệu được tải trực tiếp từ nguồn chuẩn của benchmark CKAN (GroupLens MovieLens, Stanford SNAP và tác giả repo CKAN). Code tự động tải file zip, giải nén vào thư mục `./data/` và nạp vào bộ nhớ.



In [ ]:
# ============================================================
# 4. TẢI VÀ CHUẨN BỊ DỮ LIỆU TỪ NGUỒN CHUẨN
# ============================================================
DATA_SOURCES = {
    "movie": {
        "url": "https://raw.githubusercontent.com/weberrr/CKAN/master/data/movie.zip",
        "fallback_urls": [
            "https://github.com/weberrr/CKAN/raw/master/data/movie.zip",
            "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
        ],
        "dir": "./data/movie",
        "rating_threshold": 4,
        "min_user_ratings": 10
    },
    "book": {
        "url": "https://raw.githubusercontent.com/weberrr/CKAN/master/data/book.zip",
        "fallback_urls": [
            "https://github.com/weberrr/CKAN/raw/master/data/book.zip"
        ],
        "dir": "./data/book",
        "rating_threshold": 0,
        "min_user_ratings": 5
    },
    "music": {
        "url": "https://raw.githubusercontent.com/weberrr/CKAN/master/data/music.zip",
        "fallback_urls": [
            "https://github.com/weberrr/CKAN/raw/master/data/music.zip"
        ],
        "dir": "./data/music",
        "rating_threshold": 0,
        "min_user_ratings": 5
    }
}

def download_and_extract(ds_name):
    cfg = DATA_SOURCES[ds_name]
    target_dir = cfg["dir"]
    os.makedirs(target_dir, exist_ok=True)
    
    # Kiểm tra nếu đã có file npy thì dùng luôn
    if os.path.exists(os.path.join(target_dir, "ratings_final.npy")) and os.path.exists(os.path.join(target_dir, "kg_final.npy")):
        print(f"  [OK] Tập dữ liệu {ds_name} đã sẵn sàng trong {target_dir}")
        return

    # Nếu chưa có, tải file zip
    zip_path = os.path.join(target_dir, f"{ds_name}.zip")
    urls_to_try = [cfg["url"]] + cfg["fallback_urls"]
    downloaded = False
    
    for url in urls_to_try:
        try:
            print(f"  [DOWNLOAD] Đang tải {ds_name} từ {url}...")
            urllib.request.urlretrieve(url, zip_path)
            if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000:
                downloaded = True
                print(f"  [DOWNLOAD] Tải thành công ({round(os.path.getsize(zip_path)/1e6, 2)} MB)")
                break
        except Exception as e:
            print(f"  [WARN] Không thể tải từ {url}: {e}")

    if downloaded:
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            print(f"  [UNZIP] Đã giải nén vào {target_dir}")
        except Exception as e:
            print(f"  [ERROR] Lỗi giải nén: {e}")

for ds in active_datasets:
    download_and_extract(ds)



In [ ]:
# ============================================================
# TIỀN XỬ LÝ VÀ CHUYỂN ĐỔI SANG ĐỊNH DẠNG NUMPY (.NPY)
# ============================================================
def preprocess_and_cache(ds_name):
    ds_dir = f"./data/{ds_name}"
    r_npy = os.path.join(ds_dir, "ratings_final.npy")
    k_npy = os.path.join(ds_dir, "kg_final.npy")
    
    if os.path.exists(r_npy) and os.path.exists(k_npy):
        r_test = np.load(r_npy)
        k_test = np.load(k_npy)
        if len(r_test) > 0 and len(k_test) > 0:
            print(f"  [CACHE OK] {ds_name.upper()}: Ratings={len(r_test):,} | Triples={len(k_test):,}")
            return

    print(f"  [PREPROCESS] Đang tiền xử lý dữ liệu cho {ds_name}...")
    
    # 1. Đọc ánh xạ item sang entity
    item_id2idx, entity_id2idx = {}, {}
    with open(os.path.join(ds_dir, "item_index2entity_id.txt"), "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                it_id, ent_id = parts[0], parts[1]
                idx = len(item_id2idx)
                item_id2idx[it_id] = idx
                entity_id2idx[ent_id] = idx

    # 2. Đọc ratings
    user_pos = defaultdict(set)
    user_neg = defaultdict(set)
    with open(os.path.join(ds_dir, "ratings.txt"), "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 3 and p[1] in item_id2idx:
                u, it, r = p[0], item_id2idx[p[1]], float(p[2])
                thresh = DATA_SOURCES[ds_name]["rating_threshold"]
                if thresh == 0 or r >= thresh:
                    user_pos[u].add(it)
                else:
                    user_neg[u].add(it)

    # 3. Lọc user theo min ratings
    min_u = DATA_SOURCES[ds_name]["min_user_ratings"]
    valid_users = {u: items for u, items in user_pos.items() if len(items) >= min_u}
    item_set = set()
    for items in valid_users.values(): item_set.update(items)
    
    # 4. Tạo mẫu âm tính (Negative sampling)
    all_items = list(item_set)
    rows = []
    for u_new, (u_old, pos_items) in enumerate(valid_users.items()):
        for it in pos_items: rows.append((u_new, it, 1))
        neg_candidates = list(set(all_items) - pos_items)
        if len(neg_candidates) >= len(pos_items):
            neg_items = np.random.choice(neg_candidates, size=len(pos_items), replace=False)
            for it in neg_items: rows.append((u_new, it, 0))
    rating_np = np.array(rows, dtype=np.int32)
    np.save(r_npy, rating_np)
    
    # 5. Đánh số lại KG triples
    ent_cnt = len(entity_id2idx)
    rel_id2idx, kg_rows = {}, []
    with open(os.path.join(ds_dir, "kg.txt"), "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) != 3: continue
            h_old, r_old, t_old = p[0], p[1], p[2]
            if h_old not in entity_id2idx: entity_id2idx[h_old] = ent_cnt; ent_cnt += 1
            if t_old not in entity_id2idx: entity_id2idx[t_old] = ent_cnt; ent_cnt += 1
            if r_old not in rel_id2idx: rel_id2idx[r_old] = len(rel_id2idx)
            kg_rows.append((entity_id2idx[h_old], rel_id2idx[r_old], entity_id2idx[t_old]))
            
    kg_np = np.array(kg_rows, dtype=np.int32)
    np.save(k_npy, kg_np)
    print(f"  [HOÀN TẤT {ds_name.upper()}] Users: {len(valid_users):,} | Items: {len(item_set):,} | Ratings: {len(rating_np):,} | Triples: {len(kg_np):,}")

for ds in active_datasets:
    preprocess_and_cache(ds)
print("[OK] Toàn bộ dữ liệu từ nguồn chuẩn đã sẵn sàng!")



## 5. Khám phá dữ liệu (EDA)

Phần này phân tích các đặc trưng của 3 tập dữ liệu trước khi huấn luyện mô hình:
1. **Độ thưa (Sparsity)**: Tỉ lệ ô trống trong ma trận User - Item.
2. **Phân bố tương tác (Long-tail)**: Nhóm item phổ biến chiếm bao nhiêu % lượng tương tác (nguyên lý Pareto 80/20).
3. **Đặc trưng người dùng (User Activity & Cold-start)**: Tỉ lệ user có ít hơn hoặc bằng 5 tương tác.
4. **Cấu trúc Knowledge Graph**: Số lượng entity, quan hệ, triple và phân bố bậc kết nối.

Tất cả bảng thống kê (CSV) và các biểu đồ (PNG) sẽ được lưu tự động vào thư mục `./outputs/`.



In [ ]:
# ============================================================
# 5.1. BẢNG THỐNG KÊ ĐẶC TRƯNG DỮ LIỆU (EDA SUMMARY)
# ============================================================
eda_summary = []

for ds in active_datasets:
    r_data = np.load(f"./data/{ds}/ratings_final.npy")
    k_data = np.load(f"./data/{ds}/kg_final.npy")
    
    n_users = len(np.unique(r_data[:, 0]))
    n_items = len(np.unique(r_data[:, 1]))
    n_ratings = len(r_data)
    
    # 1. Độ thưa của ma trận tương tác
    sparsity = (1.0 - n_ratings / (n_users * n_items)) * 100
    
    # 2. Phân phối tương tác theo user
    user_counts = list(Counter(r_data[:, 0]).values())
    avg_u_inter = float(np.mean(user_counts))
    med_u_inter = float(np.median(user_counts))
    cold_start_users = sum(1 for c in user_counts if c <= 5) / n_users * 100
    
    # 3. Phân phối tương tác theo item & Pareto (Top 20% item chiếm bao nhiêu % tương tác)
    item_counts = sorted(list(Counter(r_data[:, 1]).values()), reverse=True)
    cum_inter = np.cumsum(item_counts) / n_ratings * 100
    top20_share = cum_inter[min(int(0.2 * len(item_counts)), len(cum_inter) - 1)]
    
    # 4. Đặc tính Knowledge Graph
    all_entities = np.unique(np.concatenate([k_data[:, 0], k_data[:, 2]]))
    n_entities = len(all_entities)
    n_relations = len(np.unique(k_data[:, 1]))
    n_triples = len(k_data)
    branching_factor = n_triples / n_entities
    
    eda_summary.append({
        "Dataset": ds.capitalize(),
        "Users": n_users,
        "Items": n_items,
        "Ratings": n_ratings,
        "Sparsity (%)": round(sparsity, 2),
        "Avg_Inter/User": round(avg_u_inter, 1),
        "ColdStart_Users (%)": round(cold_start_users, 1),
        "Top20%_Item_Share (%)": round(top20_share, 1),
        "KG_Entities": n_entities,
        "KG_Relations": n_relations,
        "KG_Triples": n_triples,
        "KG_Branching": round(branching_factor, 2)
    })

df_eda = pd.DataFrame(eda_summary)
os.makedirs("./outputs", exist_ok=True)
df_eda.to_csv("./outputs/eda_summary.csv", index=False)

print("="*85)
print("BẢNG THỐNG KÊ ĐẶC TÍNH DỮ LIỆU & KNOWLEDGE GRAPH (EDA SUMMARY):")
print("="*85)
print(df_eda.to_string(index=False))
print("\n[OK] Đã lưu file bảng số liệu vào: ./outputs/eda_summary.csv")



In [ ]:
# ============================================================
# 5.2. TRỰC QUAN HÓA EDA VÀ LƯU HÌNH ẢNH RA FILE
# ============================================================
sample_ds = active_datasets[0]
r_sample = np.load(f"./data/{sample_ds}/ratings_final.npy")
k_sample = np.load(f"./data/{sample_ds}/kg_final.npy")

u_dist = list(Counter(r_sample[:, 0]).values())
i_dist = sorted(list(Counter(r_sample[:, 1]).values()), reverse=True)
rel_dist = Counter(k_sample[:, 1]).most_common(10)
head_dist = list(Counter(k_sample[:, 0]).values())

os.makedirs("./outputs", exist_ok=True)

# --- 1. LƯU TỪNG HÌNH ĐƠN LẺ ĐỂ TIỆN DÙNG CHO BÁO CÁO / SLIDE ---

# Hình 1: Long-Tail Item Popularity
fig_lt, ax_lt = plt.subplots(figsize=(7, 4.5), dpi=300)
ax_lt.plot(range(len(i_dist)), i_dist, color="#D97706", lw=2.5, label="Lượt tương tác của Item")
ax_lt.fill_between(range(len(i_dist)), i_dist, color="#FDE68A", alpha=0.5)
p20_idx = int(0.2 * len(i_dist))
ax_lt.axvline(p20_idx, color="#DC2626", linestyle="--", lw=1.5, label=f"Top 20% Items (Chiếm {df_eda.loc[0, 'Top20%_Item_Share (%)']}%)")
ax_lt.set_title(f"1. Phân phối Long-Tail ({sample_ds.upper()})", fontsize=11, fontweight="bold")
ax_lt.set_xlabel("Thứ hạng sản phẩm (từ phổ biến đến ngách)")
ax_lt.set_ylabel("Số lượt tương tác")
ax_lt.legend(loc="upper right")
ax_lt.grid(axis="both", linestyle=":", alpha=0.6)
fig_lt.tight_layout()
fig_lt.savefig("./outputs/eda_1_long_tail.png")
plt.close(fig_lt)

# Hình 2: User Activity Distribution
fig_ua, ax_ua = plt.subplots(figsize=(7, 4.5), dpi=300)
ax_ua.hist(u_dist, bins=35, color="#3B82F6", edgecolor="white", alpha=0.85)
ax_ua.axvline(np.median(u_dist), color="#1E3A8A", linestyle="--", lw=2, label=f"Trung vị: {int(np.median(u_dist))} tương tác")
ax_ua.set_title(f"2. Phân phối tương tác theo User ({sample_ds.upper()})", fontsize=11, fontweight="bold")
ax_ua.set_xlabel("Số lượng tương tác / user")
ax_ua.set_ylabel("Số lượng user")
ax_ua.legend(loc="upper right")
ax_ua.grid(axis="both", linestyle=":", alpha=0.6)
fig_ua.tight_layout()
fig_ua.savefig("./outputs/eda_2_user_activity.png")
plt.close(fig_ua)

# Hình 3: Top KG Relations
rel_labels = [f"Quan hệ #{r[0]}" for r in rel_dist]
rel_vals = [r[1] for r in rel_dist]
fig_rel, ax_rel = plt.subplots(figsize=(7, 4.5), dpi=300)
ax_rel.barh(rel_labels[::-1], rel_vals[::-1], color="#10B981", edgecolor="#047857", height=0.65)
ax_rel.set_title(f"3. Top 10 quan hệ phổ biến trong KG ({sample_ds.upper()})", fontsize=11, fontweight="bold")
ax_rel.set_xlabel("Số lượng bộ ba (Triples)")
ax_rel.grid(axis="x", linestyle=":", alpha=0.6)
fig_rel.tight_layout()
fig_rel.savefig("./outputs/eda_3_kg_relations.png")
plt.close(fig_rel)

# Hình 4: Scale-Free Entity Degree (Log-Log)
deg_counts = Counter(head_dist)
degs = sorted(deg_counts.keys())
freqs = [deg_counts[d] for d in degs]
fig_deg, ax_deg = plt.subplots(figsize=(7, 4.5), dpi=300)
ax_deg.scatter(degs, freqs, color="#8B5CF6", alpha=0.75, s=30, edgecolors="#6D28D9")
ax_deg.set_xscale("log")
ax_deg.set_yscale("log")
ax_deg.set_title(f"4. Bậc kết nối thực thể - Log-Log Plot ({sample_ds.upper()})", fontsize=11, fontweight="bold")
ax_deg.set_xlabel("Bậc kết nối out-degree (Log)")
ax_deg.set_ylabel("Số lượng thực thể (Log)")
ax_deg.grid(axis="both", linestyle=":", alpha=0.6)
fig_deg.tight_layout()
fig_deg.savefig("./outputs/eda_4_scale_free.png")
plt.close(fig_deg)

# --- 2. VẼ VÀ HIỂN THỊ DASHBOARD TỔNG HỢP 4 TRONG 1 ---
fig, axes = plt.subplots(2, 2, figsize=(15, 11), dpi=300)

axes[0, 0].plot(range(len(i_dist)), i_dist, color="#D97706", lw=2.5, label="Độ phổ biến sản phẩm")
axes[0, 0].fill_between(range(len(i_dist)), i_dist, color="#FDE68A", alpha=0.5)
axes[0, 0].axvline(p20_idx, color="#DC2626", linestyle="--", lw=1.5, label=f"Top 20% Items ({df_eda.loc[0, 'Top20%_Item_Share (%)']}%)")
axes[0, 0].set_title(f"1. Phân Phối Đuôi Dài (Long-Tail) - {sample_ds.upper()}", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel("Thứ hạng sản phẩm")
axes[0, 0].set_ylabel("Số lượt tương tác")
axes[0, 0].legend(loc="upper right")
axes[0, 0].grid(axis="both", linestyle=":", alpha=0.6)

axes[0, 1].hist(u_dist, bins=35, color="#3B82F6", edgecolor="white", alpha=0.85)
axes[0, 1].axvline(np.median(u_dist), color="#1E3A8A", linestyle="--", lw=2, label=f"Trung vị: {int(np.median(u_dist))}")
axes[0, 1].set_title(f"2. Tần Suất Hoạt Động Của User - {sample_ds.upper()}", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("Số tương tác / user")
axes[0, 1].set_ylabel("Số lượng user")
axes[0, 1].legend(loc="upper right")
axes[0, 1].grid(axis="both", linestyle=":", alpha=0.6)

axes[1, 0].barh(rel_labels[::-1], rel_vals[::-1], color="#10B981", edgecolor="#047857", height=0.65)
axes[1, 0].set_title(f"3. Top 10 Quan Hệ Trong KG - {sample_ds.upper()}", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("Số lượng triples")
axes[1, 0].grid(axis="x", linestyle=":", alpha=0.6)

axes[1, 1].scatter(degs, freqs, color="#8B5CF6", alpha=0.75, s=30, edgecolors="#6D28D9")
axes[1, 1].set_xscale("log")
axes[1, 1].set_yscale("log")
axes[1, 1].set_title(f"4. Bậc Kết Nối Thực Thể (Log-Log) - {sample_ds.upper()}", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel("Bậc kết nối (Log)")
axes[1, 1].set_ylabel("Số thực thể (Log)")
axes[1, 1].grid(axis="both", linestyle=":", alpha=0.6)

fig.suptitle(f"DASHBOARD KHÁM PHÁ DỮ LIỆU & ĐỒ THỊ TRI THỨC (EDA): {sample_ds.upper()}",
             fontsize=13, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("./outputs/eda_dashboard.png", bbox_inches="tight")
plt.savefig("./eda_visualization_dashboard.png", bbox_inches="tight")
plt.show()

print("[OK] Đã lưu 4 file ảnh riêng lẻ và 1 ảnh dashboard tổng hợp vào ./outputs/:")
print("  • ./outputs/eda_1_long_tail.png")
print("  • ./outputs/eda_2_user_activity.png")
print("  • ./outputs/eda_3_kg_relations.png")
print("  • ./outputs/eda_4_scale_free.png")
print("  • ./outputs/eda_dashboard.png")



### 5.3. Nhận xét thực tế từ kết quả EDA:
1. **Độ thưa rất cao (>99.4%)**:
   - Ma trận tương tác hầu hết là ô trống. Các thuật toán lọc cộng tác truyền thống (CF/MF) chỉ dựa vào dữ liệu tương tác nội sinh nên rất dễ bị thiếu dữ liệu học. Vì vậy việc bổ sung thêm Knowledge Graph là cần thiết.
2. **Hiện tượng đuôi dài (Long-tail)**:
   - Phần lớn tương tác dồn vào 20% item phổ biến nhất. Đồ thị tri thức đóng vai trò kết nối các item ít tương tác ở vùng đuôi với người dùng thông qua các thuộc tính chung (thể loại, đạo diễn, tác giả).
3. **Phân bố bậc của KG**:
   - Biểu đồ Log-Log dốc xuống chứng minh đồ thị tri thức có một số entity trung tâm (hub) có tới hàng nghìn liên kết. Nếu không lấy mẫu cố định mà duyệt toàn bộ láng giềng thì sẽ tràn bộ nhớ RAM/VRAM. Do đó CKAN dùng cơ chế lấy mẫu kích thước cố định (`itss = 64`, `utss = 32`).



## PHẦN A: CÁC MÔ HÌNH BASELINE (MostPopular, Item-KNN, Matrix Factorization)

Phần này triển khai 3 baseline cơ bản bằng các thư viện chuẩn (`scikit-learn`, `scipy`, `PyTorch`):
1. **MostPopular**: Gợi ý dựa trên tần suất xuất hiện toàn cục của item.
2. **Item-KNN**: Thuật toán láng giềng gần nhất theo độ đo Cosine dùng `NearestNeighbors` và ma trận thưa `scipy.sparse.csr_matrix`.
3. **Matrix Factorization (MF)**: Phân rã ma trận người dùng - sản phẩm viết bằng PyTorch tối giản.



In [ ]:
# ============================================================
# CÀI ĐẶT 3 MÔ HÌNH BASELINE
# ============================================================

# --- 1. MOST POPULAR BASELINE ---
class MostPopularBaseline:
    def __init__(self):
        self.item_scores = defaultdict(float)

    def fit(self, train_data, n_items):
        pos = train_data[train_data[:, 2] == 1]
        counts = Counter(pos[:, 1])
        max_c = max(counts.values()) if counts else 1.0
        for i in range(n_items):
            self.item_scores[i] = counts.get(i, 0) / max_c

    def predict(self, users, items):
        return np.array([self.item_scores.get(int(i), 0.0) for i in items])

    def score_all_items(self, u, n_items):
        return np.array([self.item_scores.get(i, 0.0) for i in range(n_items)])

# --- 2. ITEM-KNN BASELINE (sklearn NearestNeighbors + csr_matrix) ---
class ItemKNNBaseline:
    def __init__(self, k=20):
        self.k = k
        self.model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute")

    def fit(self, train_data, n_users, n_items):
        pos = train_data[train_data[:, 2] == 1]
        self.mat = csr_matrix((np.ones(len(pos)), (pos[:, 1], pos[:, 0])), shape=(n_items, n_users))
        self.model.fit(self.mat)
        self.user_history = defaultdict(set)
        for u, i in zip(pos[:, 0], pos[:, 1]):
            self.user_history[u].add(i)

    def predict(self, users, items):
        scores = []
        for u, i in zip(users, items):
            hist = self.user_history.get(u, set())
            if not hist or i >= self.mat.shape[0]:
                scores.append(0.5)
                continue
            dists, indices = self.model.kneighbors(self.mat[i], n_neighbors=min(self.k, self.mat.shape[0]))
            sims = 1.0 - dists[0]
            matched = [sims[idx] for idx, neighbor in enumerate(indices[0]) if neighbor in hist]
            scores.append(float(np.mean(matched)) if matched else 0.5)
        return np.array(scores)

# --- 3. MATRIX FACTORIZATION (PyTorch Latent Factor Model) ---
class MatrixFactorizationBaseline(nn.Module):
    def __init__(self, n_user, n_item, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        dot = (self.user_emb(users) * self.item_emb(items)).sum(dim=-1, keepdim=True)
        return torch.sigmoid((dot + self.user_bias(users) + self.item_bias(items)).squeeze(-1))

    def score_all_items(self, u, device):
        u_t = torch.LongTensor([u]).to(device)
        u_e = self.user_emb(u_t)
        u_b = self.user_bias(u_t).squeeze(-1)
        dots = torch.matmul(u_e, self.item_emb.weight.T).squeeze(0) + u_b + self.item_bias.weight.squeeze(-1)
        return dots.detach().cpu().numpy()

print("[OK] Đã hoàn tất cài đặt 3 mô hình Baseline!")



## PHẦN B: CÁC MODULE CỦA MÔ HÌNH CKAN

Mô hình CKAN được chia thành 6 module độc lập:
1. **`KnowledgeRippleSampler`**: Lấy mẫu láng giềng (Ripple Sets) trên đồ thị tri thức qua $L$ tầng cho cả User và Item.
2. **`KnowledgeAwareAttentionLayer`**: Tính trọng số chú ý giữa relation và thực thể.
3. **`CKANAggregator`**: Gom thông tin láng giềng lại với node gốc (hỗ trợ Concat / Sum / Neighbor).
4. **`CKAN`**: Mạng tổng hợp kết hợp hai nhánh Collaborative User và Semantic Item.
5. **`CKANTrainer`**: Huấn luyện mô hình với Binary Cross-Entropy loss và L2 regularization.
6. **`TopKRecommenderEvaluator`**: Đánh giá xếp hạng Top-K (Recall@K, NDCG@K, Precision@K) tối ưu tính toán trên GPU.



In [ ]:
# ============================================================
# B1. KNOWLEDGE RIPPLE SAMPLER (LẤY MẪU LÁNG GIỀNG TRÊN ĐỒ THỊ)
# ============================================================
class KnowledgeRippleSampler:
    def __init__(self, kg_np, n_layer=1, itss=64, utss=32):
        self.n_layer = n_layer
        self.itss = itss
        self.utss = utss
        
        self.kg_dict = defaultdict(list)
        for h, r, t in kg_np:
            self.kg_dict[int(h)].append((int(t), int(r)))

    def build_item_ripple_set(self, n_items):
        item_triple_set = defaultdict(list)
        for it in range(n_items):
            for l in range(self.n_layer):
                neighbors = self.kg_dict.get(it, [])
                if len(neighbors) == 0:
                    h, r, t = [it] * self.itss, [0] * self.itss, [it] * self.itss
                else:
                    replace = len(neighbors) < self.itss
                    choice = np.random.choice(len(neighbors), size=self.itss, replace=replace)
                    h = [it] * self.itss
                    r = [neighbors[c][1] for c in choice]
                    t = [neighbors[c][0] for c in choice]
                item_triple_set[it].append((h, r, t))
        return item_triple_set

    def build_user_ripple_set(self, user_history_dict):
        user_triple_set = defaultdict(list)
        for u, history in user_history_dict.items():
            current_heads = list(history)
            for l in range(self.n_layer):
                h_list, r_list, t_list = [], [], []
                for h in current_heads:
                    for t, r in self.kg_dict.get(h, []):
                        h_list.append(h); r_list.append(r); t_list.append(t)
                if len(h_list) == 0:
                    fallback_h = list(history)[0] if len(history) > 0 else 0
                    h, r, t = [fallback_h] * self.utss, [0] * self.utss, [fallback_h] * self.utss
                else:
                    replace = len(h_list) < self.utss
                    choice = np.random.choice(len(h_list), size=self.utss, replace=replace)
                    h = [h_list[c] for c in choice]
                    r = [r_list[c] for c in choice]
                    t = [t_list[c] for c in choice]
                user_triple_set[u].append((h, r, t))
                current_heads = t
        return user_triple_set

print("[OK] Mô-đun B1: KnowledgeRippleSampler đã sẵn sàng!")



In [ ]:
# ============================================================
# B2. KNOWLEDGE-AWARE ATTENTION LAYER
# ============================================================
class KnowledgeAwareAttentionLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1)
        )

    def forward(self, head_emb, rel_emb, tail_emb):
        # head_emb: [batch_size, n_samples, dim]
        # rel_emb : [batch_size, n_samples, dim]
        # tail_emb: [batch_size, n_samples, dim]
        hr_concat = torch.cat([head_emb, rel_emb], dim=-1)
        scores = self.mlp(hr_concat).squeeze(-1)
        attn_weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        context_vec = (attn_weights * tail_emb).sum(dim=1)
        return context_vec, attn_weights

print("[OK] Mô-đun B2: KnowledgeAwareAttentionLayer đã sẵn sàng!")



In [ ]:
# ============================================================
# B3. INFORMATION AGGREGATOR LAYER
# ============================================================
class CKANAggregator(nn.Module):
    def __init__(self, dim, agg_type="concat"):
        super().__init__()
        self.dim = dim
        self.agg_type = agg_type
        if agg_type == "concat":
            self.linear = nn.Linear(dim * 2, dim)
        elif agg_type == "sum":
            self.linear = nn.Linear(dim, dim)
        elif agg_type == "neighbor":
            self.linear = nn.Linear(dim, dim)

    def forward(self, self_vec, neighbor_vec):
        if self.agg_type == "concat":
            combined = torch.cat([self_vec, neighbor_vec], dim=-1)
            return F.relu(self.linear(combined))
        elif self.agg_type == "sum":
            return F.relu(self.linear(self_vec + neighbor_vec))
        elif self.agg_type == "neighbor":
            return F.relu(self.linear(neighbor_vec))

print("[OK] Mô-đun B3: CKANAggregator đã sẵn sàng!")



In [ ]:
# ============================================================
# B4. MÔ HÌNH CKAN (COLLABORATIVE KNOWLEDGE ATTENTIVE NETWORK)
# ============================================================
class CKAN(nn.Module):
    def __init__(self, n_entity, n_relation, dim=64, n_layer=1, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        
        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
        self.attn_layers = nn.ModuleList([KnowledgeAwareAttentionLayer(dim) for _ in range(n_layer)])
        self.user_aggs = nn.ModuleList([CKANAggregator(dim, agg) for _ in range(n_layer)])
        self.item_aggs = nn.ModuleList([CKANAggregator(dim, agg) for _ in range(n_layer)])

    def _propagate_knowledge(self, root_emb, triple_set, is_user=True):
        current_rep = root_emb
        for l in range(self.n_layer):
            h, r, t = triple_set[l]
            h_e = self.entity_emb(h)
            r_e = self.relation_emb(r)
            t_e = self.entity_emb(t)
            context, _ = self.attn_layers[l](h_e, r_e, t_e)
            agg = self.user_aggs[l] if is_user else self.item_aggs[l]
            current_rep = agg(current_rep, context)
        return current_rep

    def forward(self, items, user_triples, item_triples):
        it_e = self.entity_emb(items)
        it_rep = self._propagate_knowledge(it_e, item_triples, is_user=False)
        u_init = self.entity_emb(user_triples[0][0]).mean(dim=1)
        u_rep = self._propagate_knowledge(u_init, user_triples, is_user=True)
        scores = (u_rep * it_rep).sum(dim=-1)
        return torch.sigmoid(scores)

    def get_item_embeddings(self, item_ids, item_triple_set, device):
        self.eval()
        with torch.no_grad():
            it_tensor = torch.LongTensor(item_ids).to(device)
            it_e = self.entity_emb(it_tensor)
            triples = []
            for l in range(self.n_layer):
                h = torch.LongTensor([item_triple_set[i][l][0] for i in item_ids]).to(device)
                r = torch.LongTensor([item_triple_set[i][l][1] for i in item_ids]).to(device)
                t = torch.LongTensor([item_triple_set[i][l][2] for i in item_ids]).to(device)
                triples.append((h, r, t))
            return self._propagate_knowledge(it_e, triples, is_user=False)

    def get_user_embeddings(self, user_triple_tuples):
        self.eval()
        with torch.no_grad():
            u_init = self.entity_emb(user_triple_tuples[0][0]).mean(dim=1)
            return self._propagate_knowledge(u_init, user_triple_tuples, is_user=True)

print("[OK] Mô-đun B4: Kiến trúc mô hình CKAN đã hoàn tất!")



In [ ]:
# ============================================================
# B5. CKAN TRAINER
# ============================================================
class CKANTrainer:
    def __init__(self, model, lr=0.002, weight_decay=1e-5, device="cuda"):
        self.model = model.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.BCELoss()

    def train_epoch(self, train_data, user_triples_dict, item_triples_dict, n_layer, batch_size=2048):
        self.model.train()
        np.random.shuffle(train_data)
        total_loss = 0.0
        n_batches = 0
        
        for start in range(0, len(train_data), batch_size):
            batch = train_data[start:start+batch_size]
            u_batch = batch[:, 0]
            i_batch = batch[:, 1]
            labels = torch.FloatTensor(batch[:, 2]).to(self.device)
            
            u_triples = []
            i_triples = []
            for l in range(n_layer):
                uh = torch.LongTensor([user_triples_dict[u][l][0] for u in u_batch]).to(self.device)
                ur = torch.LongTensor([user_triples_dict[u][l][1] for u in u_batch]).to(self.device)
                ut = torch.LongTensor([user_triples_dict[u][l][2] for u in u_batch]).to(self.device)
                u_triples.append((uh, ur, ut))
                
                ih = torch.LongTensor([item_triples_dict[i][l][0] for i in i_batch]).to(self.device)
                ir = torch.LongTensor([item_triples_dict[i][l][1] for i in i_batch]).to(self.device)
                it = torch.LongTensor([item_triples_dict[i][l][2] for i in i_batch]).to(self.device)
                i_triples.append((ih, ir, it))
                
            self.optimizer.zero_grad()
            preds = self.model(torch.LongTensor(i_batch).to(self.device), u_triples, i_triples)
            loss = self.criterion(preds, labels)
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
            
        return total_loss / max(1, n_batches)

    def evaluate(self, test_data, user_triples_dict, item_triples_dict, n_layer, batch_size=2048):
        self.model.eval()
        all_preds = []
        with torch.no_grad():
            for start in range(0, len(test_data), batch_size):
                batch = test_data[start:start+batch_size]
                u_batch = batch[:, 0]
                i_batch = batch[:, 1]
                
                u_triples = []
                i_triples = []
                for l in range(n_layer):
                    uh = torch.LongTensor([user_triples_dict[u][l][0] for u in u_batch]).to(self.device)
                    ur = torch.LongTensor([user_triples_dict[u][l][1] for u in u_batch]).to(self.device)
                    ut = torch.LongTensor([user_triples_dict[u][l][2] for u in u_batch]).to(self.device)
                    u_triples.append((uh, ur, ut))
                    
                    ih = torch.LongTensor([item_triples_dict[i][l][0] for i in i_batch]).to(self.device)
                    ir = torch.LongTensor([item_triples_dict[i][l][1] for i in i_batch]).to(self.device)
                    it = torch.LongTensor([item_triples_dict[i][l][2] for i in i_batch]).to(self.device)
                    i_triples.append((ih, ir, it))
                    
                preds = self.model(torch.LongTensor(i_batch).to(self.device), u_triples, i_triples)
                all_preds.extend(preds.cpu().numpy())
                
        all_preds = np.array(all_preds)
        labels = test_data[:, 2]
        auc = roc_auc_score(labels, all_preds)
        f1 = f1_score(labels, (all_preds >= 0.5).astype(int), zero_division=0)
        acc = accuracy_score(labels, (all_preds >= 0.5).astype(int))
        return auc, f1, acc

print("[OK] Mô-đun B5: CKANTrainer đã sẵn sàng!")



In [ ]:
# ============================================================
# B6. TOP-K RECOMMENDER EVALUATOR (RECALL@K, NDCG@K, PRECISION@K)
# ============================================================
class TopKRecommenderEvaluator:
    def __init__(self, k_list=[5, 10, 20]):
        self.k_list = k_list
        self.max_k = max(k_list)

    def evaluate_model(self, score_func, eval_users, train_pos_dict, test_pos_dict, n_items):
        metrics = {f"Recall@{k}": [] for k in self.k_list}
        metrics.update({f"NDCG@{k}": [] for k in self.k_list})
        metrics.update({f"Precision@{k}": [] for k in self.k_list})
        
        for u in eval_users:
            test_pos = test_pos_dict.get(u, set())
            if not test_pos: continue
            
            scores = score_func(u)
            train_pos = train_pos_dict.get(u, set())
            if train_pos:
                scores[list(train_pos)] = -1e9
                
            top_items = np.argpartition(scores, -self.max_k)[-self.max_k:]
            top_items = top_items[np.argsort(-scores[top_items])]
            
            for k in self.k_list:
                top_k = top_items[:k]
                hits = sum(1 for it in top_k if it in test_pos)
                
                # Recall@K
                metrics[f"Recall@{k}"].append(hits / len(test_pos))
                # Precision@K
                metrics[f"Precision@{k}"].append(hits / k)
                
                # NDCG@K
                dcg = sum(1.0 / np.log2(idx + 2) for idx, it in enumerate(top_k) if it in test_pos)
                idcg = sum(1.0 / np.log2(idx + 2) for idx in range(min(k, len(test_pos))))
                metrics[f"NDCG@{k}"].append(dcg / idcg if idcg > 0 else 0.0)
                
        return {m: float(np.mean(vals)) if vals else 0.0 for m, vals in metrics.items()}

print("[OK] Mô-đun B6: TopKRecommenderEvaluator đã sẵn sàng!")



## PHẦN C: CHẠY BENCHMARK TRÊN 3 TẬP DỮ LIỆU

Tiến trình chạy tuần tự trên cả 3 tập dữ liệu (**Movie**, **Book**, **Music**):
1. **Dự đoán tương tác (CTR Prediction)**: So sánh AUC, F1 và Accuracy giữa 4 mô hình.
2. **Xếp hạng danh sách Top-K**: Đánh giá Recall@10 và NDCG@10.
3. **Thử nghiệm giảm dữ liệu (Sparsity Stress Test)**: Giữ lại 10% tập train để xem khi thiếu dữ liệu thì MF sụt bao nhiêu và CKAN duy trì ra sao.
4. **Xuất bảng kết quả**: Lưu ra `./outputs/benchmark_results.csv`.



In [ ]:
# ============================================================
# C1. TIẾN HÀNH BENCHMARK TRÊN 3 TẬP DỮ LIỆU
# ============================================================
def evaluate_predictions(labels, scores):
    auc = roc_auc_score(labels, scores)
    f1 = f1_score(labels, (scores >= 0.5).astype(int), zero_division=0)
    acc = accuracy_score(labels, (scores >= 0.5).astype(int))
    return auc, f1, acc

def run_comprehensive_benchmark(ds_name):
    print(f"\n{'='*75}")
    print(f">>> BẮT ĐẦU CHẠY BENCHMARK CHO: {ds_name.upper()} <<<")
    print(f"{'='*75}")
    
    cfg = CONFIG[ds_name]
    rating_np = np.load(f"./data/{ds_name}/ratings_final.npy")
    kg_np = np.load(f"./data/{ds_name}/kg_final.npy")
    
    n_user = len(np.unique(rating_np[:, 0]))
    n_item = len(np.unique(rating_np[:, 1]))
    all_ents = np.unique(np.concatenate([kg_np[:, 0], kg_np[:, 2]]))
    n_entity = max(len(all_ents), n_item)
    n_relation = len(np.unique(kg_np[:, 1]))
    
    print(f"Dataset: {ds_name.capitalize()} | Users: {n_user:,} | Items: {n_item:,} | Ratings: {len(rating_np):,} | KG Triples: {len(kg_np):,}")
    
    # Chia tập train (80%) và test (20%)
    np.random.shuffle(rating_np)
    split = int(len(rating_np) * 0.8)
    train_data = rating_np[:split]
    test_data = rating_np[split:]
    
    train_pos_dict = defaultdict(set)
    for u, i, r in train_data:
        if r == 1: train_pos_dict[u].add(i)
    test_pos_dict = defaultdict(set)
    for u, i, r in test_data:
        if r == 1: test_pos_dict[u].add(i)
        
    eval_users = [u for u in test_pos_dict if len(test_pos_dict[u]) >= 3][:200]
    topk_evaluator = TopKRecommenderEvaluator(k_list=[5, 10, 20])
    
    # ----------------------------------------------------
    # 1. MOST POPULAR
    # ----------------------------------------------------
    pop = MostPopularBaseline()
    pop.fit(train_data, n_item)
    pop_scores = pop.predict(test_data[:, 0], test_data[:, 1])
    pop_auc, pop_f1, _ = evaluate_predictions(test_data[:, 2], pop_scores)
    pop_topk = topk_evaluator.evaluate_model(lambda u: pop.score_all_items(u, n_item), eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [1/4] MostPopular       : AUC={pop_auc:.4f} | F1={pop_f1:.4f} | Recall@10={pop_topk['Recall@10']:.4f} | NDCG@10={pop_topk['NDCG@10']:.4f}")
    
    # ----------------------------------------------------
    # 2. ITEM-KNN (sklearn NearestNeighbors)
    # ----------------------------------------------------
    knn = ItemKNNBaseline(k=20)
    knn.fit(train_data, n_user, n_item)
    knn_scores = knn.predict(test_data[:, 0], test_data[:, 1])
    knn_auc, knn_f1, _ = evaluate_predictions(test_data[:, 2], knn_scores)
    print(f"  [2/4] Item-KNN (sklearn): AUC={knn_auc:.4f} | F1={knn_f1:.4f}")
    
    # ----------------------------------------------------
    # 3. MATRIX FACTORIZATION (Biased MF PyTorch)
    # ----------------------------------------------------
    mf = MatrixFactorizationBaseline(n_user, n_item, dim=cfg["dim"]).to(device)
    opt_mf = torch.optim.Adam(mf.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    crit = nn.BCELoss()
    bs = cfg["batch_size"]
    
    for ep in range(6):
        mf.train()
        for s in range(0, len(train_data), bs):
            b = train_data[s:s+bs]
            opt_mf.zero_grad()
            pred = mf(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device))
            loss = crit(pred, torch.FloatTensor(b[:, 2]).to(device))
            loss.backward()
            opt_mf.step()
            
    mf.eval()
    with torch.no_grad():
        mf_scores = [mf(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [test_data[s:s+bs] for s in range(0, len(test_data), bs)]]
        mf_scores = np.concatenate(mf_scores)
    mf_auc, mf_f1, _ = evaluate_predictions(test_data[:, 2], mf_scores)
    mf_topk = topk_evaluator.evaluate_model(lambda u: mf.score_all_items(u, device), eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [3/4] Matrix Factorization: AUC={mf_auc:.4f} | F1={mf_f1:.4f} | Recall@10={mf_topk['Recall@10']:.4f} | NDCG@10={mf_topk['NDCG@10']:.4f}")
    
    # ----------------------------------------------------
    # 4. CKAN (COLLABORATIVE KNOWLEDGE ATTENTION)
    # ----------------------------------------------------
    sampler = KnowledgeRippleSampler(kg_np, n_layer=cfg["n_layer"], itss=cfg["itss"], utss=cfg["utss"])
    item_triple_set = sampler.build_item_ripple_set(n_item)
    user_triple_set = sampler.build_user_ripple_set(train_pos_dict)
    
    ckan = CKAN(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"], agg="concat")
    trainer = CKANTrainer(ckan, lr=cfg["lr"], weight_decay=cfg["l2_weight"], device=device)
    
    for ep in range(cfg["n_epochs"]):
        trainer.train_epoch(train_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
        
    ckan_auc, ckan_f1, _ = trainer.evaluate(test_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
    
    # Đánh giá Top-K cho CKAN
    all_item_matrix = []
    chunk_size = 2048
    for ch_s in range(0, n_item, chunk_size):
        ch_ids = list(range(ch_s, min(ch_s + chunk_size, n_item)))
        all_item_matrix.append(ckan.get_item_embeddings(ch_ids, item_triple_set, device))
    all_item_matrix = torch.cat(all_item_matrix, dim=0)
    
    def ckan_score_fn(u):
        with torch.no_grad():
            u_tr = []
            for l in range(cfg["n_layer"]):
                uh = torch.LongTensor([user_triple_set[u][l][0]]).to(device)
                ur = torch.LongTensor([user_triple_set[u][l][1]]).to(device)
                ut = torch.LongTensor([user_triple_set[u][l][2]]).to(device)
                u_tr.append((uh, ur, ut))
            u_emb = ckan.get_user_embeddings(u_tr)
            scores = torch.matmul(u_emb, all_item_matrix.T).squeeze(0)
            return scores.cpu().numpy()
            
    ckan_topk = topk_evaluator.evaluate_model(ckan_score_fn, eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [4/4] CKAN (With KG)    : AUC={ckan_auc:.4f} | F1={ckan_f1:.4f} | Recall@10={ckan_topk['Recall@10']:.4f} | NDCG@10={ckan_topk['NDCG@10']:.4f}")
    
    # ----------------------------------------------------
    # 5. SPARSITY TEST (10% DATA HUẤN LUYỆN)
    # ----------------------------------------------------
    sub_tr = train_data[:int(len(train_data) * 0.1)]
    
    # MF Sparse
    mf_sp = MatrixFactorizationBaseline(n_user, n_item, dim=cfg["dim"]).to(device)
    opt_sp = torch.optim.Adam(mf_sp.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    for _ in range(4):
        mf_sp.train()
        for s in range(0, len(sub_tr), bs):
            b = sub_tr[s:s+bs]
            opt_sp.zero_grad()
            crit(mf_sp(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)), torch.FloatTensor(b[:, 2]).to(device)).backward()
            opt_sp.step()
    with torch.no_grad():
        mf_sp_sc = [mf_sp(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [test_data[s:s+bs] for s in range(0, len(test_data), bs)]]
        mf_sp_sc = np.concatenate(mf_sp_sc)
    mf_sp_auc, _, _ = evaluate_predictions(test_data[:, 2], mf_sp_sc)
    
    # CKAN Sparse
    ck_sp = CKAN(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"], agg="concat")
    trainer_sp = CKANTrainer(ck_sp, lr=cfg["lr"], weight_decay=1e-5, device=device)
    for _ in range(4):
        trainer_sp.train_epoch(sub_tr, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
    ck_sp_auc, _, _ = trainer_sp.evaluate(test_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
    
    diff = (ck_sp_auc - mf_sp_auc) * 100
    print(f"  [Sparsity 10%] MF AUC = {mf_sp_auc:.4f} | CKAN AUC = {ck_sp_auc:.4f} (CKAN vượt trội: +{diff:.1f}%)")
    
    return {
        "Dataset": ds_name.capitalize(),
        "Users": n_user, "Items": n_item, "Ratings": len(rating_np), "KG_Triples": len(kg_np),
        "MostPop_AUC": pop_auc, "ItemKNN_AUC": knn_auc, "MF_AUC": mf_auc, "CKAN_AUC": ckan_auc,
        "MF_F1": mf_f1, "CKAN_F1": ckan_f1,
        "MostPop_NDCG10": pop_topk['NDCG@10'], "MF_NDCG10": mf_topk['NDCG@10'], "CKAN_NDCG10": ckan_topk['NDCG@10'],
        "MostPop_Rec10": pop_topk['Recall@10'], "MF_Rec10": mf_topk['Recall@10'], "CKAN_Rec10": ckan_topk['Recall@10'],
        "MF_Sparse10_AUC": mf_sp_auc, "CKAN_Sparse10_AUC": ck_sp_auc
    }

results = []
for ds in active_datasets:
    results.append(run_comprehensive_benchmark(ds))

df_results = pd.DataFrame(results)
os.makedirs("./outputs", exist_ok=True)
df_results.to_csv("./outputs/benchmark_results.csv", index=False)

print("\n" + "="*80)
print("BẢNG TỔNG KẾT KẾT QUẢ BENCHMARK TRÊN 3 TẬP DỮ LIỆU:")
print("="*80)
cols_display = ["Dataset", "Users", "Items", "Ratings", "MF_AUC", "CKAN_AUC", "MF_NDCG10", "CKAN_NDCG10", "MF_Rec10", "CKAN_Rec10", "MF_Sparse10_AUC", "CKAN_Sparse10_AUC"]
print(df_results[cols_display].to_string(index=False))
print("\n[OK] Đã lưu bảng kết quả benchmark ra file: ./outputs/benchmark_results.csv")



In [ ]:
# ============================================================
# C2. TRỰC QUAN HÓA KẾT QUẢ & LƯU TẤT CẢ BIỂU ĐỒ RA FILE
# ============================================================
os.makedirs("./outputs", exist_ok=True)
x = np.arange(len(df_results))
width = 0.20
w2 = 0.25

# --- 1. LƯU TỪNG HÌNH BENCHMARK ĐƠN LẺ ---

# Hình 1: So sánh ROC-AUC
fig1, ax1 = plt.subplots(figsize=(7, 4.5), dpi=300)
ax1.bar(x - 1.5*width, df_results["MostPop_AUC"], width, label="MostPopular", color="#9CA3AF")
ax1.bar(x - 0.5*width, df_results["ItemKNN_AUC"], width, label="Item-KNN (sklearn)", color="#60A5FA")
ax1.bar(x + 0.5*width, df_results["MF_AUC"], width, label="Biased MF", color="#3B82F6")
ax1.bar(x + 1.5*width, df_results["CKAN_AUC"], width, label="CKAN (Proposed)", color="#D97706")
ax1.set_xticks(x)
ax1.set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
ax1.set_ylabel("ROC-AUC")
ax1.set_title("1. So Sánh ROC-AUC (CTR Prediction)", fontsize=11, fontweight="bold")
ax1.set_ylim(0.4, 1.05)
ax1.legend(loc="lower right")
ax1.grid(axis="y", linestyle=":", alpha=0.7)
fig1.tight_layout()
fig1.savefig("./outputs/benchmark_1_ctr_auc.png")
plt.close(fig1)

# Hình 2: So sánh NDCG@10
fig2, ax2 = plt.subplots(figsize=(7, 4.5), dpi=300)
ax2.bar(x - w2, df_results["MostPop_NDCG10"], w2, label="MostPopular", color="#9CA3AF")
ax2.bar(x, df_results["MF_NDCG10"], w2, label="Biased MF", color="#3B82F6")
ax2.bar(x + w2, df_results["CKAN_NDCG10"], w2, label="CKAN (Proposed)", color="#D97706")
ax2.set_xticks(x)
ax2.set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
ax2.set_ylabel("NDCG@10")
ax2.set_title("2. Hiệu Suất Xếp Hạng Top-10 (NDCG@10)", fontsize=11, fontweight="bold")
ax2.legend(loc="upper right")
ax2.grid(axis="y", linestyle=":", alpha=0.7)
fig2.tight_layout()
fig2.savefig("./outputs/benchmark_2_topk_ndcg10.png")
plt.close(fig2)

# Hình 3: So sánh Recall@10
fig3, ax3 = plt.subplots(figsize=(7, 4.5), dpi=300)
ax3.bar(x - w2, df_results["MostPop_Rec10"], w2, label="MostPopular", color="#9CA3AF")
ax3.bar(x, df_results["MF_Rec10"], w2, label="Biased MF", color="#3B82F6")
ax3.bar(x + w2, df_results["CKAN_Rec10"], w2, label="CKAN (Proposed)", color="#D97706")
ax3.set_xticks(x)
ax3.set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
ax3.set_ylabel("Recall@10")
ax3.set_title("3. Độ Phủ Nhu Cầu Người Dùng (Recall@10)", fontsize=11, fontweight="bold")
ax3.legend(loc="upper right")
ax3.grid(axis="y", linestyle=":", alpha=0.7)
fig3.tight_layout()
fig3.savefig("./outputs/benchmark_3_topk_recall10.png")
plt.close(fig3)

# Hình 4: Sparsity Stress Test (10% Data)
fig4, ax4 = plt.subplots(figsize=(7, 4.5), dpi=300)
w3 = 0.35
ax4.bar(x - w3/2, df_results["MF_Sparse10_AUC"], w3, label="MF (10% Data)", color="#93C5FD", edgecolor="#3B82F6")
ax4.bar(x + w3/2, df_results["CKAN_Sparse10_AUC"], w3, label="CKAN (10% Data)", color="#F59E0B", edgecolor="#D97706")
ax4.set_xticks(x)
ax4.set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
ax4.set_ylabel("ROC-AUC")
ax4.set_title("4. Thử Nghiệm Độ Thưa (10% Training Data)", fontsize=11, fontweight="bold", color="#991B1B")
ax4.set_ylim(0.4, 1.05)
ax4.legend(loc="lower right")
ax4.grid(axis="y", linestyle=":", alpha=0.7)
for i in range(len(df_results)):
    diff = (df_results["CKAN_Sparse10_AUC"][i] - df_results["MF_Sparse10_AUC"][i]) * 100
    ax4.text(i, max(df_results["CKAN_Sparse10_AUC"][i], df_results["MF_Sparse10_AUC"][i]) + 0.03, f"+{diff:.1f}%",
             ha="center", va="bottom", fontsize=10, fontweight="bold", color="#B45309")
fig4.tight_layout()
fig4.savefig("./outputs/benchmark_4_sparsity_test.png")
plt.close(fig4)

# --- 2. VẼ VÀ HIỂN THỊ DASHBOARD TỔNG HỢP 4 TRONG 1 ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)

axes[0, 0].bar(x - 1.5*width, df_results["MostPop_AUC"], width, label="MostPopular", color="#9CA3AF")
axes[0, 0].bar(x - 0.5*width, df_results["ItemKNN_AUC"], width, label="Item-KNN (sklearn)", color="#60A5FA")
axes[0, 0].bar(x + 0.5*width, df_results["MF_AUC"], width, label="Biased MF", color="#3B82F6")
axes[0, 0].bar(x + 1.5*width, df_results["CKAN_AUC"], width, label="CKAN (KG Attention)", color="#D97706")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
axes[0, 0].set_ylabel("ROC-AUC", fontsize=11, fontweight="bold")
axes[0, 0].set_title("1. So Sánh ROC-AUC (Dự Đoán CTR)", fontsize=12, fontweight="bold")
axes[0, 0].set_ylim(0.4, 1.05)
axes[0, 0].legend(loc="lower right")
axes[0, 0].grid(axis="y", linestyle=":", alpha=0.7)

axes[0, 1].bar(x - w2, df_results["MostPop_NDCG10"], w2, label="MostPopular", color="#9CA3AF")
axes[0, 1].bar(x, df_results["MF_NDCG10"], w2, label="Biased MF", color="#3B82F6")
axes[0, 1].bar(x + w2, df_results["CKAN_NDCG10"], w2, label="CKAN (Proposed)", color="#D97706")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
axes[0, 1].set_ylabel("NDCG@10", fontsize=11, fontweight="bold")
axes[0, 1].set_title("2. Hiệu Suất Xếp Hạng Top-10 (NDCG@10)", fontsize=12, fontweight="bold")
axes[0, 1].legend(loc="upper right")
axes[0, 1].grid(axis="y", linestyle=":", alpha=0.7)

axes[1, 0].bar(x - w2, df_results["MostPop_Rec10"], w2, label="MostPopular", color="#9CA3AF")
axes[1, 0].bar(x, df_results["MF_Rec10"], w2, label="Biased MF", color="#3B82F6")
axes[1, 0].bar(x + w2, df_results["CKAN_Rec10"], w2, label="CKAN (Proposed)", color="#D97706")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("Recall@10", fontsize=11, fontweight="bold")
axes[1, 0].set_title("3. Độ Phủ Nhu Cầu Người Dùng (Recall@10)", fontsize=12, fontweight="bold")
axes[1, 0].legend(loc="upper right")
axes[1, 0].grid(axis="y", linestyle=":", alpha=0.7)

bars1 = axes[1, 1].bar(x - w3/2, df_results["MF_Sparse10_AUC"], w3, label="MF (10% Data)", color="#93C5FD", edgecolor="#3B82F6")
bars2 = axes[1, 1].bar(x + w3/2, df_results["CKAN_Sparse10_AUC"], w3, label="CKAN (10% Data)", color="#F59E0B", edgecolor="#D97706")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
axes[1, 1].set_ylabel("ROC-AUC", fontsize=11, fontweight="bold")
axes[1, 1].set_title("4. Thử Nghiệm Độ Thưa (10% Training Data)", fontsize=12, fontweight="bold", color="#991B1B")
axes[1, 1].set_ylim(0.4, 1.05)
axes[1, 1].legend(loc="lower right")
axes[1, 1].grid(axis="y", linestyle=":", alpha=0.7)

for i in range(len(df_results)):
    diff = (df_results["CKAN_Sparse10_AUC"][i] - df_results["MF_Sparse10_AUC"][i]) * 100
    axes[1, 1].text(i, max(df_results["CKAN_Sparse10_AUC"][i], df_results["MF_Sparse10_AUC"][i]) + 0.03, f"+{diff:.1f}%",
                    ha="center", va="bottom", fontsize=10, fontweight="bold", color="#B45309")

fig.suptitle("DASHBOARD TỔNG HỢP KẾT QUẢ BENCHMARK (CTR PREDICTION & TOP-K RANKING)",
             fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("./outputs/benchmark_dashboard.png", bbox_inches="tight")
plt.savefig("./tri_dataset_scientific_dashboard.png", bbox_inches="tight")
plt.show()

print("[OK] Đã lưu 4 biểu đồ đơn lẻ và 1 dashboard tổng hợp vào ./outputs/:")
print("  • ./outputs/benchmark_1_ctr_auc.png")
print("  • ./outputs/benchmark_2_topk_ndcg10.png")
print("  • ./outputs/benchmark_3_topk_recall10.png")
print("  • ./outputs/benchmark_4_sparsity_test.png")
print("  • ./outputs/benchmark_dashboard.png")



## PHẦN D: TỔNG KẾT VÀ GIẢI THÍCH KẾT QUẢ

Từ kết quả đo đạc thực tế ở trên, có 4 điểm đáng chú ý:

### 1. Vì sao MostPopular có AUC cao nhưng F1 và Top-K lại thấp?
- **Hiện tượng**: MostPopular đạt ROC-AUC rất cao (~0.96 trên Movie, ~0.75 trên Book), ngang ngửa với các mô hình học sâu.
- **Lý do**: ROC-AUC đo lường thứ hạng tương đối (xác suất mẫu positive được chấm điểm cao hơn mẫu negative ngẫu nhiên). Do các item hot chiếm đa số lượt xem trong tập test, việc luôn xếp item hot lên đầu giúp MostPopular thắng phần lớn các phép so sánh cặp.
- **Thực tế**: Khi đo bằng **F1-Score** và **Top-K (NDCG@10, Recall@10)**, MostPopular rớt thê thảm (F1 chỉ đạt 0.25 trên Movie và 0.07 trên Book) vì mô hình không hề cá nhân hóa mà gợi ý danh sách giống hệt nhau cho mọi user.

### 2. Vì sao Item-KNN bị tụt khi ma trận thưa?
- Thuật toán Item-KNN tính độ tương đồng Cosine giữa các cột sản phẩm trong ma trận tương tác.
- Với độ thưa >99.4% (Movie) và >99.9% (Book), xác suất hai item bất kỳ cùng được đánh giá bởi cùng một nhóm user là rất thấp.
- Vì không tìm thấy láng giềng phù hợp, mô hình đành gán điểm mặc định 0.5, khiến AUC tụt xuống mức 0.26 - 0.48.

### 3. Vì sao CKAN ổn định hơn MF khi chỉ còn 10% data?
- Khi cắt giảm dữ liệu huấn luyện xuống 10% (mô phỏng tình huống thiếu dữ liệu tương tác):
  - **Matrix Factorization**: Bị mất tới **-12.5% AUC** (từ 0.9617 xuống 0.8366 trên MovieLens) do không có đủ tương tác để học embedding.
  - **CKAN**: Đạt **0.9465 AUC**, chỉ suy giảm nhẹ 1.7% và **vượt hơn MF +11.0%**.
- **Nguyên nhân**: Dù ít tương tác giữa user và item, CKAN vẫn có gần 500,000 bộ ba tri thức (triples) từ Knowledge Graph. Mạng Attention lan truyền sở thích qua các thực thể liên quan (đạo diễn, diễn viên, thể loại) để bù đắp cho lượng rating bị thiếu.

### 4. Khi nào nên dùng CKAN?
- Nếu hệ thống đã có hàng triệu lượt tương tác dày đặc, mô hình Matrix Factorization đơn giản đã đủ tốt ($AUC \approx 0.96$).
- Knowledge Graph và CKAN phát huy hiệu quả rõ nhất khi:
  - Hệ thống mới mở hoặc dữ liệu còn thưa thớt (Cold-start).
  - Cần gợi ý các item ở vùng đuôi dài (Long-tail) mà CF thông thường không với tới được.
  - Cần giải thích lý do gợi ý cho người dùng dựa vào các đường dẫn trên đồ thị tri thức.



In [ ]:
# ============================================================
# KIỂM TRA TOÀN BỘ FILE ĐÃ XUẤT RA (CSV & HÌNH ẢNH)
# ============================================================
import glob

print("Danh sách các file kết quả và biểu đồ đã lưu trong ./outputs/:")
files = sorted(glob.glob("./outputs/*"))
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  • {f} ({size_kb:.1f} KB)")

# Ví dụ đọc lại bảng số liệu trong code:
# import pandas as pd
# df_eda = pd.read_csv("./outputs/eda_summary.csv")
# df_bench = pd.read_csv("./outputs/benchmark_results.csv")
# print(df_bench.head())

